# DexGen: Automated Render Node
This notebook serves as the remote inference server for the DexGen Distributed Pipeline.

### 🚀 Quick Start
1. Ensure **Runtime > Change runtime type** is set to **T4 GPU**.
2. Go to **Runtime > Run all**.
3. Wait for the **Server is ready** message at the bottom.

## 1. Environment Setup

In [ ]:
import os
from google.colab import drive

print("📦 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

ROOT = "/content/drive/MyDrive/DexGen"
os.makedirs(f"{ROOT}/inputs", exist_ok=True)
os.makedirs(f"{ROOT}/outputs", exist_ok=True)
print(f"✅ Workspace ready at {ROOT}")

In [ ]:
!apt-get update -y > /dev/null
!apt-get install -y ffmpeg > /dev/null
!wget -q -O cloudflared-linux-amd64.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null
!pip install -q -U diffusers transformers accelerate safetensors peft sentencepiece fastapi uvicorn python-multipart pillow imageio imageio-ffmpeg nest_asyncio
print("✅ Dependencies installed.")

## 2. Secrets & Security

In [ ]:
import json
SECRET_PATH = "/content/drive/MyDrive/DexGen/secrets.json"
DEFAULT_KEY = "starsilk_remote_auth_key"

if not os.path.exists(SECRET_PATH):
    print("🔑 secrets.json not found. Creating with default key...")
    with open(SECRET_PATH, "w") as f:
        json.dump({"DEXGEN_API_KEY": DEFAULT_KEY}, f)

with open(SECRET_PATH, "r") as f:
    secrets_data = json.load(f)
    API_KEY = secrets_data.get("DEXGEN_API_KEY", DEFAULT_KEY)

print(f"✅ API Key loaded: {API_KEY[:4]}**** (ensure this matches your macOS client)")

## 3. Backend Engine

In [ ]:
import time
import os
import json
import asyncio
import secrets as py_secrets
import gc
import threading
import subprocess
import re
import torch
import imageio
import numpy as np
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import uvicorn
import nest_asyncio
from PIL import Image

app = FastAPI()
STATUS_FILE = "/content/drive/MyDrive/DexGen/status.json"
URL_FILE = "/content/drive/MyDrive/DexGen/current_url.txt"

# Ensure API_KEY is accessible in this cell if Cell 2 was skipped during a restart
SECRET_PATH = "/content/drive/MyDrive/DexGen/secrets.json"
DEFAULT_KEY = "starsilk_remote_auth_key"
try:
    with open(SECRET_PATH, "r") as f:
        API_KEY = json.load(f).get("DEXGEN_API_KEY", DEFAULT_KEY)
except FileNotFoundError:
    API_KEY = DEFAULT_KEY

inference_lock = threading.Lock()
active_pipeline = None
active_model_id = None

global_status = {
    "ok": False,
    "base_url": None,
    "started_at": int(time.time()),
    "last_error": "Initializing"
}

def update_status(ok: bool, error: str = None, base_url: str = None):
    global_status["ok"] = ok
    if error is not None:
        global_status["last_error"] = error
    elif ok:
        global_status["last_error"] = None
    if base_url is not None:
        global_status["base_url"] = base_url
    try:
        with open(STATUS_FILE, "w") as f:
            json.dump(global_status, f)
    except: pass

def clear_current_url():
    try:
        os.remove(URL_FILE)
    except FileNotFoundError:
        pass

class ModelManager:
    @staticmethod
    def cleanup():
        global active_pipeline, active_model_id
        if active_pipeline is not None:
            del active_pipeline
            active_pipeline = None
            active_model_id = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    @staticmethod
    def load(model_id: str):
        global active_pipeline, active_model_id
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA not available. Use a T4/A100 runtime in Colab.")

        if active_model_id == model_id and active_pipeline is not None:
            return active_pipeline

        ModelManager.cleanup()
        print(f"Loading model: {model_id}")

        if model_id == "sd15":
            from diffusers import StableDiffusionPipeline
            pipe = StableDiffusionPipeline.from_pretrained(
                "runwayml/stable-diffusion-v1-5",
                torch_dtype=torch.float16, use_safetensors=True
            ).to("cuda")
        elif model_id == "flux_schnell":
            from diffusers import FluxPipeline
            # float16 avoids bfloat16 emulation path on T4; FLUX can still OOM on free-tier memory limits.
            pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.float16)
            pipe.enable_model_cpu_offload()
        elif model_id == "i2vgen_xl":
            from diffusers import I2VGenXLPipeline
            pipe = I2VGenXLPipeline.from_pretrained("ali-vilab/i2vgen-xl", torch_dtype=torch.float16, variant="fp16")
            pipe.enable_model_cpu_offload()
        elif model_id == "svd":
            from diffusers import StableVideoDiffusionPipeline
            pipe = StableVideoDiffusionPipeline.from_pretrained("stabilityai/stable-video-diffusion-img2vid-xt", torch_dtype=torch.float16, variant="fp16")
            pipe.enable_model_cpu_offload()
        else:
            raise ValueError(f"Unknown model: {model_id}")

        active_model_id = model_id
        active_pipeline = pipe
        return pipe

def write_mp4(frames, path, fps):
    writer = imageio.get_writer(path, fps=fps, codec="libx264", format="FFMPEG")
    try:
        for f in frames:
            writer.append_data(np.array(f.convert("RGB") if isinstance(f, Image.Image) else f))
    finally:
        writer.close()

@app.middleware("http")
async def verify_api_key(request: Request, call_next):
    if request.headers.get("X-API-Key") != API_KEY:
        return JSONResponse(status_code=401, content={"error": "Unauthorized"})
    return await call_next(request)

@app.get("/")
def health():
    update_status(True, None, global_status.get("base_url"))
    return {"ok": True, "version": "1.0.2", "models": ["sd15", "flux_schnell", "i2vgen_xl", "svd"]}

@app.post("/generate")
async def generate_image(req: Request):
    try:
        data = await req.json()

        def _do_inference():
            model_id = data.get("model", "sd15")
            seed = int(data.get("seed", 0)) or py_secrets.randbelow(2**31-1)

            with inference_lock:
                pipe = ModelManager.load(model_id)
                gen = torch.Generator("cuda").manual_seed(seed)

                if model_id == "sd15":
                    out = pipe(prompt=data["prompt"], negative_prompt=data.get("negative_prompt"), num_inference_steps=data.get("steps", 30), width=int(data.get("width", 512)), height=int(data.get("height", 512)), guidance_scale=data.get("guidance_scale", 7.5), generator=gen).images[0]
                else:
                    out = pipe(prompt=data["prompt"], guidance_scale=0.0, num_inference_steps=data.get("steps", 30), max_sequence_length=256, height=int(data.get("height", 512)), width=int(data.get("width", 512)), generator=gen).images[0]

                fname = f"img_{model_id}_{seed}_{int(time.time())}.png"
                path = f"/content/drive/MyDrive/DexGen/outputs/{fname}"
                out.save(path)
                update_status(True, None, global_status.get("base_url"))
                return {"saved_to": path, "seed": seed}

        # Offload blocking GPU work from the event loop.
        return await asyncio.to_thread(_do_inference)
    except Exception as e:
        update_status(bool(global_status.get("base_url")), str(e), global_status.get("base_url"))
        return JSONResponse(status_code=500, content={"error": str(e)})

@app.post("/generate_video")
async def generate_video(req: Request):
    try:
        data = await req.json()
        img_path = data.get("image_path")
        if not img_path or not os.path.exists(img_path):
            raise FileNotFoundError(f"Source missing: {img_path}")

        def _do_inference():
            model_id = data.get("model", "i2vgen_xl")
            seed = int(data.get("seed", 0)) or py_secrets.randbelow(2**31-1)

            with inference_lock:
                pipe = ModelManager.load(model_id)
                gen = torch.Generator("cuda").manual_seed(seed)
                base_img = Image.open(img_path).convert("RGB")

                if model_id == "i2vgen_xl":
                    frames = pipe(prompt=data.get("prompt", ""), image=base_img.resize((512, 512)), num_inference_steps=data.get("steps", 30), num_frames=data.get("frames", 16), generator=gen).frames[0]
                else:
                    frames = pipe(image=base_img.resize((1024, 576)), num_frames=data.get("frames", 16), num_inference_steps=data.get("steps", 25), decode_chunk_size=8, generator=gen).frames[0]

                fname = f"vid_{model_id}_{seed}_{int(time.time())}.mp4"
                path = f"/content/drive/MyDrive/DexGen/outputs/{fname}"
                write_mp4(frames, path, data.get("fps", 8))
                update_status(True, None, global_status.get("base_url"))
                return {"saved_to": path, "seed": seed}

        # Offload blocking GPU work from the event loop.
        return await asyncio.to_thread(_do_inference)
    except Exception as e:
        update_status(bool(global_status.get("base_url")), str(e), global_status.get("base_url"))
        return JSONResponse(status_code=500, content={"error": str(e)})

print("✅ Backend Logic Loaded.")



## 4. Execution

In [ ]:
nest_asyncio.apply()
clear_current_url()
update_status(False, "Starting API server")
server_thread = threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info"), daemon=True)
server_thread.start()
time.sleep(2)

print("🌐 Starting cloudflared...")
process = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(90):
    line = process.stdout.readline()
    if "https://" in line and ".trycloudflare.com" in line:
        public_url = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', line).group(1)
        break
    time.sleep(1)

if public_url:
    with open(URL_FILE, "w") as f: f.write(public_url)
    update_status(True, None, public_url)
    print(f"\n{'='*40}\n🚀 SERVER IS READY\n🔗 URL: {public_url}\n{'='*40}\n")
    process.wait()
else:
    process.terminate()
    update_status(False, "Cloudflare tunnel did not produce a public URL in time.")
    print("❌ Cloudflare failed to start.")